# 📦 E-Commerce Sales Data Processing and Analysis
## Using Apache Spark RDDs · Python · PySpark

This notebook walks through **every important RDD operation** demonstrated in the project pipeline.
Each section includes:
- A plain-English explanation of the operation
- The PySpark RDD code
- Live output from your dataset

---
**Prerequisites:** Run `python scripts/run_pipeline.py` first to generate and clean the dataset.

## 0. Setup – Imports and SparkSession

In [ ]:
import os, sys

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')   # notebooks/ is one level below root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.spark_session import get_spark_session
from src.utils.logger import load_config

spark = get_spark_session(app_name='NB_Demo', log_level='ERROR')
sc    = spark.sparkContext

cfg      = load_config()
RAW_PATH = os.path.join(PROJECT_ROOT, cfg['paths']['raw_data'])
OUT_DIR  = os.path.join(PROJECT_ROOT, cfg['paths']['output_dir'])

print(f'Spark version : {spark.version}')
print(f'Raw data path : {RAW_PATH}')
print(f'Output dir    : {OUT_DIR}')

## 1. Loading the Raw CSV into an RDD

`sc.textFile()` reads the CSV line-by-line into an RDD of strings.
The **header** line is identified with `first()` and removed with `filter()`.

In [ ]:
raw_rdd = sc.textFile(RAW_PATH)
header  = raw_rdd.first()            # first() → inspect first element
print('Header:', header)

data_rdd = raw_rdd.filter(lambda line: line != header)   # filter()
print(f'Total data rows (excl header): {data_rdd.count():,}')  # count()

## 2. `map()` — Parse CSV Lines into Dicts

`map()` applies a function to **every element** of the RDD, transforming each raw CSV string into a structured Python dict.

In [ ]:
COLUMNS = [
    'transaction_id','date','customer_id','product_id','product_name',
    'category','quantity','unit_price','discount','payment_method',
    'city','state','order_status','rating'
]

def parse_row(line):
    parts = line.split(',')
    if len(parts) != len(COLUMNS):
        return None
    row = dict(zip(COLUMNS, [p.strip() for p in parts]))
    try: row['quantity']   = int(row['quantity'])
    except: row['quantity'] = None
    try: row['unit_price'] = float(row['unit_price'])
    except: row['unit_price'] = None
    try: row['discount']   = float(row['discount']) if row['discount'] else 0.0
    except: row['discount'] = None
    try: row['rating']     = float(row['rating']) if row['rating'] else None
    except: row['rating']  = None
    return row

parsed_rdd = data_rdd.map(parse_row)   # ← map()

sample = parsed_rdd.take(2)            # take() → first 2 elements
for rec in sample:
    print(rec)

## 3. `filter()` — Remove Invalid Rows

`filter()` keeps only elements for which the function returns `True`.
We reject rows with: `None` result, negative prices, zero quantity, out-of-range discounts/ratings, or empty required fields.

In [ ]:
def is_valid(row):
    if row is None: return False
    required = ['transaction_id','date','customer_id','product_name',
                'category','payment_method','city','state','order_status']
    for f in required:
        if not row.get(f): return False
    if (row.get('quantity') or 0) < 1:       return False
    if (row.get('unit_price') or 0) <= 0:    return False
    if row.get('discount') is None or not (0.0 <= row['discount'] <= 1.0): return False
    if row.get('rating') is not None and not (1.0 <= row['rating'] <= 5.0): return False
    return True

valid_rdd = parsed_rdd.filter(is_valid)   # ← filter()

before = parsed_rdd.count()
after  = valid_rdd.count()
print(f'Before filter : {before:,}')
print(f'After  filter : {after:,}  (removed {before-after:,} invalid rows)')

## 4. `distinct()` — Remove Duplicate Rows

`distinct()` returns a new RDD with **unique elements only**.
Since dicts aren't hashable directly, we convert each row to a canonical string first.

In [ ]:
deduped_rdd = (
    valid_rdd
    .map(lambda r: str(sorted(r.items())))   # dict → string
    .distinct()                              # ← distinct()
    .map(lambda s: dict(eval(s)))            # string → dict
)

print(f'After deduplication: {deduped_rdd.count():,}')
print(f'Duplicates removed : {after - deduped_rdd.count():,}')

## 5. `cache()` — Materialise the RDD in Memory

`cache()` tells Spark to **keep this RDD in memory** after it's first computed.
Without it, each action would re-read and re-process the file from scratch.

In [ ]:
# Add revenue column and cache
cleaned_rdd = deduped_rdd.map(
    lambda r: {**r, 'revenue': round(r['quantity'] * r['unit_price'] * (1 - r['discount']), 2)}
).cache()   # ← cache()

print(f'Cleaned & cached RDD count: {cleaned_rdd.count():,}')
print('\nSample record:')
print(cleaned_rdd.first())

## 6. `reduce()` — Compute Total Revenue

`reduce()` aggregates all elements of the RDD down to a **single value** using a binary function.
Here we sum all revenue values.

In [ ]:
total_revenue = cleaned_rdd.map(lambda r: r['revenue']).reduce(lambda a, b: a + b)   # ← reduce()
total_qty     = cleaned_rdd.map(lambda r: r['quantity']).reduce(lambda a, b: a + b)
total_txn     = cleaned_rdd.count()

print(f'Total Revenue       : ${total_revenue:>14,.2f}')
print(f'Total Transactions  : {total_txn:>15,}')
print(f'Total Units Sold    : {total_qty:>15,}')
print(f'Average Order Value : ${total_revenue/total_txn:>14,.2f}')

## 7. `reduceByKey()` — Revenue by Category

`reduceByKey()` groups records by key and applies a reduction function to the values.
It's more efficient than `groupByKey()` because it pre-aggregates on each partition.

In [ ]:
import pandas as pd

rev_by_cat = (
    cleaned_rdd
    .map(lambda r: (r['category'], r['revenue']))   # map → (key, value)
    .reduceByKey(lambda a, b: a + b)               # ← reduceByKey()
    .sortByKey()                                    # sortByKey()
    .collect()                                      # collect()
)

df_cat = pd.DataFrame(rev_by_cat, columns=['Category', 'Revenue ($)'])
df_cat['Revenue ($)'] = df_cat['Revenue ($)'].round(2)
df_cat

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,5))
df_sorted = df_cat.sort_values('Revenue ($)')
ax.barh(df_sorted['Category'], df_sorted['Revenue ($)'], color='#89b4fa')
ax.set_title('Revenue by Category', fontsize=14)
ax.set_xlabel('Revenue ($)')
plt.tight_layout()
plt.savefig('rev_by_cat.png', dpi=100)
plt.show()
print('Chart saved as rev_by_cat.png')

## 8. `sortByKey()` — Ranking Products

`sortByKey()` sorts an RDD of (key, value) pairs by key.
We negate the revenue to get **descending** order.

In [ ]:
top10_products = (
    cleaned_rdd
    .map(lambda r: (r['product_name'], r['revenue']))
    .reduceByKey(lambda a, b: a + b)
    .map(lambda kv: (-kv[1], kv[0]))    # negate revenue for desc sort
    .sortByKey()                         # ← sortByKey()
    .take(10)                            # take(10)
)

df_top10 = pd.DataFrame(
    [{'Rank': i+1, 'Product': name, 'Revenue ($)': round(-rev, 2)}
     for i, (rev, name) in enumerate(top10_products)]
)
df_top10

## 9. `groupByKey()` — Payment Method Analysis

`groupByKey()` collects all values for each key into an iterable.
Best used when you need multiple statistics per group.

In [ ]:
payment_groups = (
    cleaned_rdd
    .map(lambda r: (r['payment_method'], r['revenue']))
    .groupByKey()     # ← groupByKey()
    .mapValues(list)  # convert ResultIterable to list
    .collect()
)

rows = []
for method, revenues in payment_groups:
    rows.append({
        'Payment Method': method,
        'Transactions': len(revenues),
        'Total Revenue ($)': round(sum(revenues), 2),
        'Avg Order Value ($)': round(sum(revenues)/len(revenues), 2)
    })

pd.DataFrame(rows).sort_values('Total Revenue ($)', ascending=False)

## 10. `flatMap()` — Monthly Revenue (Two Metrics in One Pass)

`flatMap()` transforms each record into **multiple output records**.
Here, each transaction emits two key-value pairs: one for revenue, one for count.

In [ ]:
def emit_monthly(r):
    ym = r['date'][:7]   # 'YYYY-MM'
    return [
        (ym + '__revenue', r['revenue']),
        (ym + '__count',   1.0),
    ]

monthly_raw = (
    cleaned_rdd
    .flatMap(emit_monthly)              # ← flatMap() – 2 items per record
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
    .collect()
)

monthly = {}
for key, val in monthly_raw:
    ym, metric = key.split('__')
    monthly.setdefault(ym, {})[metric] = val

df_monthly = pd.DataFrame([
    {'Month': ym, 'Revenue ($)': round(v.get('revenue',0),2), 'Transactions': int(v.get('count',0))}
    for ym, v in sorted(monthly.items())
])
df_monthly.head(12)

## 11. `union()` — Merge Completed vs. Pending Orders

`union()` combines two RDDs into one.  
We split into completed (Delivered/Shipped) and pending (rest), compute stats on each, then rejoin.

In [ ]:
completed = cleaned_rdd.filter(lambda r: r['order_status'] in {'Delivered','Shipped'})
pending   = cleaned_rdd.filter(lambda r: r['order_status'] not in {'Delivered','Shipped'})

def status_pairs(rdd):
    return (
        rdd
        .map(lambda r: (r['order_status'], (r['revenue'], 1)))
        .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
    )

all_status = status_pairs(completed).union(status_pairs(pending))   # ← union()

result = all_status.collect()
df_status = pd.DataFrame([
    {'Status': s, 'Revenue ($)': round(rv,2), 'Count': cnt}
    for s,(rv,cnt) in result
]).sort_values('Revenue ($)', ascending=False)
df_status

## 12. `distinct()` — Unique Products & Customers

In [ ]:
unique_products  = cleaned_rdd.map(lambda r: r['product_name']).distinct().count()   # ← distinct()
unique_customers = cleaned_rdd.map(lambda r: r['customer_id']).distinct().count()
unique_cities    = cleaned_rdd.map(lambda r: r['city']).distinct().count()

print(f'Unique products  : {unique_products}')
print(f'Unique customers : {unique_customers:,}')
print(f'Unique cities    : {unique_cities}')

## 13. `first()` and `take()` — Data Inspection

In [ ]:
# first() → single element
first_record = cleaned_rdd.first()   # ← first()
print('first() →')
for k, v in first_record.items():
    print(f'  {k:20s}: {v}')

print()
# take(5) → first 5 elements as a list
sample_5 = cleaned_rdd.take(5)       # ← take()
print(f'take(5) → {len(sample_5)} records returned')
pd.DataFrame(sample_5)[['transaction_id','product_name','category','revenue','order_status']]

## 14. `collect()` — Pull All Results to Driver

In [ ]:
# collect() brings the entire RDD to the driver as a Python list
# ⚠ Only safe when the result is small (e.g. after aggregation)
city_revenue = (
    cleaned_rdd
    .map(lambda r: (r['city'], r['revenue']))
    .reduceByKey(lambda a, b: a + b)
    .map(lambda kv: (-kv[1], kv[0]))
    .sortByKey()
    .collect()    # ← collect()
)

df_city = pd.DataFrame(
    [{'City': city, 'Revenue ($)': round(-rev, 2)} for rev, city in city_revenue]
).head(10)
df_city

## 15. Load Pre-Generated Charts

In [ ]:
from IPython.display import Image, display

charts = [
    'chart_revenue_by_category.png',
    'chart_top_10_products.png',
    'chart_monthly_revenue.png',
    'chart_payment_analysis.png',
    'chart_order_status.png',
    'chart_top_states.png',
]

for chart in charts:
    path = os.path.join(OUT_DIR, chart)
    if os.path.exists(path):
        print(f'\n--- {chart} ---')
        display(Image(filename=path, width=800))
    else:
        print(f'[NOT FOUND] {path} – run the pipeline first')

## 16. Shutdown

In [ ]:
spark.stop()
print('SparkSession stopped.')